# TBD Phase 2: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: Single-node processing speed, parallel execution, and memory usage.
- **Scalability**: How performance changes with the number of cores (single-node) and executors (cluster).
- **Computing Models**: Out-of-core vs. In-memory processing, and Eager vs. Lazy execution.

### Engine Capabilities
The following table summarizes the key capabilities of the engines we will be testing. Use this as a reference.

| Engine | Query Optimizer | Distributed | Arrow-backed | Out-of-Core | Parallel | APIs | GPU Support |
|---|---|---|---|---|--|---|---|
| **Pandas** | ❌ | ❌ | optional ≥ 2.0 | ❌ | ❌ | DataFrame | ❌ |
| **Polars** | ✅ | ❌ | ✅ | ✅ | ✅ | DataFrame | ✅ (opt) |
| **PySpark** | ✅ | ✅ | Pandas UDF/IO | ✅ | ✅ | SQL, DataFrame | ❌ (no GPU) |
| **DuckDB** | ✅ | ❌ | ✅ | ✅ | ❌ | SQL, Relational API | ❌ |

## Prerequisites
Ensure you have the necessary libraries installed.

In [1]:
%pip install polars pandas duckdb pyspark faker deltalake memory_profiler pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 1.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.3/455.3 MB 2.9 MB/s eta 0:00:0000:0100:03
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.4/802.4 kB 6.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 MB 2.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 3.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 5.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 7.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 3.6 MB/s eta 0:00:0000:0100:01m


In [36]:
import polars as pl
import pandas as pd
import duckdb
from pyspark.sql import SparkSession
from faker import Faker
import numpy as np
import os
import time
import psutil
from memory_profiler import memory_usage

# Initialize Spark (Single Node)
spark = SparkSession.builder \
    .appName("BigDataLab2") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

26/01/06 00:41:39 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Part 1: Data Generation

We will generate a synthetic dataset simulating social media posts with a rich schema.

**Schema**:
- `post_id` (String): Unique identifier.
- `user_id` (Integer): User identifier.
- `timestamp` (DateTime): Time of post.
- `content` (String): Text content.
- `likes` (Integer): Number of likes.
- `views` (Integer): Number of views.
- `category` (String): Post category.
- `tags` (List[String]): Hashtags.
- `location` (String): User location.
- `device` (String): Device used (Mobile, Web, etc.).
- `latency` (Float): Network latency.
- `error_rate` (Float): Error rate during upload.

In [2]:
def generate_data(num_records=1_000_000, output_path="social_media_data.parquet"):
    fake = Faker()
    
    print(f"Generating {num_records} records...")
    
    # Generate data using numpy for speed where possible
    data = {
        "post_id": [fake.uuid4() for _ in range(num_records)],
        "user_id": np.random.randint(1, 100_000, num_records),
        "timestamp": pd.date_range(start="2023-01-01", periods=num_records, freq="s").to_numpy().astype("datetime64[us]"),
        "likes": np.random.randint(0, 10_000, num_records),
        "views": np.random.randint(0, 1_000_000, num_records),
        "category": np.random.choice(["Tech", "Health", "Travel", "Food", "Fashion", "Politics", "Sports"], num_records),
        "tags": [np.random.choice(["#viral", "#new", "#trending", "#hot", "#update"], size=np.random.randint(1, 4)).tolist() for _ in range(num_records)],
        "location": np.random.choice(["USA", "UK", "DE", "PL", "FR", "JP", "BR"], num_records),
        "device": np.random.choice(["Mobile", "Desktop", "Tablet"], num_records),
        "latency": np.random.uniform(10.0, 500.0, num_records),
        "error_rate": np.random.beta(1, 10, num_records),
        "content": [fake.sentence() for _ in range(min(num_records, 1000))] * (num_records // 1000 + 1)
    }
    
    users_count = 100_000
    users = {
        "user_id": np.arange(1, users_count + 1),
        "age": np.random.randint(18, 50, users_count),
        "nationality": np.random.choice(["USA", "UK", "DE", "PL", "FR", "JP", "BR"], users_count),
    }

    # Trim to exact size
    data["content"] = data["content"][:num_records]
    
    df = pd.DataFrame(data)
    
    print("Writing to Parquet...")
    df.to_parquet(output_path, engine="pyarrow")
    print(f"Data saved to {output_path}")

    df_users = pd.DataFrame(users)
    users_output_path = "users.parquet"
    df_users.to_parquet(users_output_path, engine="pyarrow")
    print(f"Users' data saved to {users_output_path}")


# Generate 5 million records
generate_data(num_records=500_000)

Generating 500000 records...
Writing to Parquet...
Data saved to social_media_data.parquet
Users' data saved to users.parquet


## Part 2: Measuring Performance

### 2.1 Execution Time
Use `%time` or `%timeit` to measure execution time.

In [39]:
# Example: Measuring time for all engines
print("--- Performance Benchmark Example ---")

# Pandas
print("Pandas Load Time:")
%time df_pd = pd.read_parquet("social_media_data.parquet")

# Polars
print("\nPolars Load Time:")
%time df_pl = pl.read_parquet("social_media_data.parquet")

# DuckDB
print("\nDuckDB Query Time:")
%time duckdb.sql("SELECT count(*) FROM 'social_media_data.parquet'").show()

# PySpark
print("\nSpark Load Time:")
%time df_spark = spark.read.parquet("social_media_data.parquet"); df_spark.count()


--- Performance Benchmark Example ---
Pandas Load Time:
CPU times: user 756 ms, sys: 208 ms, total: 964 ms
Wall time: 792 ms

Polars Load Time:
CPU times: user 115 ms, sys: 325 ms, total: 440 ms
Wall time: 1.82 s

DuckDB Query Time:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       500000 │
└──────────────┘

CPU times: user 7.93 ms, sys: 0 ns, total: 7.93 ms
Wall time: 15.9 ms

Spark Load Time:
CPU times: user 2.54 ms, sys: 17 μs, total: 2.55 ms
Wall time: 235 ms


500000

## Part 3: Student Tasks

### Task 1: Performance & Scalability (Single Node)

**Goal**: Benchmark the engines and test how they scale with available CPU cores.

**Instructions**:
1.  **Define Queries**: Create 3 distinct queries of your own choice. They should cover:
    -   **Query A**: A simple aggregation (e.g., grouping by a categorical column and calculating means).
    -   **Query B**: A window function or more complex transformation.
    -   **Query C**: A join (e.g., self-join or join with a smaller generated table) with filtering.
2.  **Benchmark**: Implement these queries in **Pandas, Polars, DuckDB, and PySpark**.
    -   Measure **Execution Time** using `%time` or `time.time()`.
    -   Measure **Peak Memory** usage using `memory_profiler` (e.g., `memory_usage()`).
3.  **Scalability Test**: 
    -   Select **all engines** that support parallel execution on a single node (e.g., Polars, DuckDB).
    -   Run **all 3 queries** with different numbers of threads/cores (e.g., 1, 2, 4, 8).
    -   Plot the speedup for each query and engine.

**Tip**: 
-   Polars: [polars.thread_pool_size](https://docs.pola.rs/api/python/stable/reference/api/polars.thread_pool_size.html) Please also note that *Thread configuration in Polars requires process restart*
-   DuckDB: `PRAGMA threads=n`
-   Spark: `master="local[n]"`

In [3]:
# Example: Measuring time for all engines
print("--- Reading users data ---")

# Pandas
print("Pandas Load Time:")
%time df_pd_users = pd.read_parquet("users.parquet")

# Polars
print("\nPolars Load Time:")
%time df_pl_users = pl.read_parquet("users.parquet")

# PySpark
print("\nSpark Load Time:")
%time df_spark_users = spark.read.parquet("users.parquet"); df_spark.count()

--- Reading users data ---
Pandas Load Time:
CPU times: user 22.7 ms, sys: 12.6 ms, total: 35.4 ms
Wall time: 44.6 ms

Polars Load Time:
CPU times: user 4.32 ms, sys: 4.07 ms, total: 8.39 ms
Wall time: 46 ms

Spark Load Time:
CPU times: user 4.56 ms, sys: 0 ns, total: 4.56 ms
Wall time: 692 ms


500000

**Query A**

Group by location, aggregate function - mean of likes

In [26]:
from pyspark.sql import functions as fn

def query_A_pandas():
    return df_pd.groupby('location')['likes'].agg('mean').shape[0]

def query_A_polars():
    return df_pl.group_by('location').agg(pl.col('likes').mean()).height

def query_A_duckdb():
    return duckdb.sql("SELECT COUNT(*) from (SELECT location, AVG(likes) FROM 'social_media_data.parquet' GROUP BY location)").fetchall()[0][0]

def query_A_spark():
    return df_spark.groupBy("location").agg(fn.avg("likes")).count()


**Query B**


**Query C**

Join posts and users by user_id. Then, filter by age (>= 25).

In [ ]:
def query_C_pandas():
    return df_pd.merge(df_pd_users, on="user_id", how="inner").query("age >= 25").shape[0]

def query_C_polars():
    return df_pl.join(df_pl_users, on="user_id", how="inner").filter(pl.col("age") >= 25).height

def query_C_duckdb():
    return duckdb.sql("SELECT COUNT(*) FROM (SELECT * FROM 'social_media_data.parquet' AS posts " \
        "INNER JOIN 'users.parquet' AS users " \
        "on posts.user_id=users.user_id where users.age >= 25)").fetchall()[0][0]

def query_C_spark():
    return df_spark.join(df_spark_users, on="user_id", how="inner").where("age >= 25").count()


390542

Benchmark - time + memory peak

In [35]:
queries = {
    "A pandas": query_A_pandas,
    "A polars": query_A_polars,
    "A duckdb": query_A_duckdb,
    "A spark": query_A_spark,

    "C pandas": query_C_pandas,
    "C polars": query_C_polars,
    "C duckdb": query_C_duckdb,
    "C spark": query_C_spark,
}


def measure_time(query):
    start = time.perf_counter()
    result = query()
    elapsed = time.perf_counter() - start
    return elapsed


def measure_peak_memory(query):
    return memory_usage(query, max_usage=True)

def benchmark(queries):
    results = []
    for query_name, query_fn in queries.items():
        # three query execution - first warm-up
        query_fn()

        # second for time benchmark
        t = measure_time(query_fn)

        # third for memory peak benchmark
        m = measure_peak_memory(query_fn)

        results.append({
            "Query": query_name,
            "Time [s]": t,
            "Memory peak [MiB]": m
        })

    return pd.DataFrame(results)

benchmark_results = benchmark(queries)
benchmark_results

,Query,Time [s],Memory peak [MiB]
0,A pandas,0.036643,846.308594
1,A polars,0.010787,859.351562
2,A duckdb,0.014582,861.050781
3,A spark,0.223338,855.445312
4,C pandas,0.223450,946.878906
5,C polars,0.206945,971.703125
6,C duckdb,0.052828,904.304688
7,C spark,0.286377,883.292969


### Task 2: Spark on Cluster

**Goal**: Compare Single Node performance vs. Spark on a Cluster.

**Instructions**:
1.  **Infrastructure**: Use the infrastructure from **Phase 1** (Google Dataproc). You may need to modify your Terraform code to adjust the cluster configuration (e.g., number of worker nodes).
2.  **Environment**: The easiest way to run this is via **Google Workbench** connected to your Dataproc cluster.
3.  **Upload Data**: Upload the generated `social_media_data.parquet` to HDFS or GCS.
    -   **Tip**: For better performance, consider **partitioning** the data (e.g., by `category` or `date`) when saving it to the distributed storage. This allows Spark to optimize reads.
4.  **Run Queries**: Run your PySpark queries from Task 1 on the cluster.
5.  **Scalability Test**: 
    -   Run the queries with different numbers of **worker nodes** (e.g., 2, 3, 4).
    -   You can achieve this by resizing the cluster (manually or via Terraform) or by configuring the number of executors in Spark.
6.  **Analyze**:
    -   How does the cluster performance compare to your local machine?
    -   Did adding more nodes/executors linearly improve performance?
    -   **Tip**: If Spark is slower than single-node engines, consider **increasing the dataset size** (e.g., generate 10M+ records or duplicate the data). Spark's overhead is significant for small data, and its true power appears when data exceeds single-node memory.

In [ ]:
# Your Code Here for Task 2

### Task 3: Execution Modes & Analysis

**Goal**: Deep dive into execution models and limitations.

**Instructions**:
1.  **Lazy vs. Eager vs. Streaming**:
    -   Use **Polars**. Compare the **Execution Time** and **Peak Memory** of:
        -   Eager execution (`read_parquet` -> filter).
        -   Lazy execution (`scan_parquet` -> filter -> `collect()`).
        -   Streaming execution (`scan_parquet` -> filter -> `collect(streaming=True)`).
2.  **Polars Limitations**:
    -   Identify a scenario where Polars might struggle compared to Spark (e.g., memory limits).
3.  **Decision Boundary**:
    -   Based on your findings, when would you recommend switching from a single-node tool (Polars/DuckDB) to a distributed engine (Spark)?

In [ ]:
# Your Code Here for Task 3